# 準備資料與環境


In [ ]:
!pip install yfinance -q

In [ ]:
import pandas as pd
import yfinance as yf

# 單檔股票
df = yf.Ticker("NVDA").history(period="5y")

# 多檔股票收盤價
tickers = ["NVDA", "AAPL", "GOOGL", "MSFT", "JPM", "WMT", "KO", "SBUX"]
price = yf.download(tickers, period="5y", progress=False)["Close"]

## 繪圖環境設定

In [ ]:
!apt-get update -qq
!apt-get install -y fonts-noto-cjk -qq

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font = FontProperties(fname=font_path, size=12)

# 特徵工程

In [ ]:
df = df.reset_index()[["Date", "Open", "High", "Low", "Close", "Volume"]]
df["Date"] = df["Date"].dt.tz_localize(None)
df = df.sort_values("Date").reset_index(drop=True)
df.tail()

## 報酬率
- **DailyReturn**：每日報酬率，抓「漲跌幅度」
- **Volatility_5**：5 日報酬率的標準差，抓「波動程度」

In [ ]:
df["DailyReturn"] = df["Close"].pct_change()
df["Volatility_5"] = df["DailyReturn"].rolling(5).std()
df[["Date", "Close", "DailyReturn", "Volatility_5"]].tail()

## [MA](https://rich01.com/what-is-moving-average-line/)
MA 代表過去一段時間裡的平均成交價格，  
最主要目的是用來判斷趨勢，通常是預期市場現在跟未來可能的走勢。

In [ ]:
df["MA_5"] = df["Close"].rolling(5).mean()
df["MA_20"] = df["Close"].rolling(20).mean()
df[["Date", "Close", "MA_5", "MA_20"]].tail()

### [練習] 算出 10 日均線 `MA_10`


In [ ]:
df["MA_10"] = df["Close"].rolling(10).mean()
df[["Date", "Close", "MA_5", "MA_10", "MA_20"]].tail()

## [RSI](https://zh.wikipedia.org/zh-tw/%E7%9B%B8%E5%B0%8D%E5%BC%B7%E5%BC%B1%E6%8C%87%E6%95%B8)

RSI 用來衡量近期漲多還是跌多，數值介於 0~100：
- 漲的力道遠大於跌 → RSI 接近 70（超買）
- 跌的力道遠大於漲 → RSI 接近 30（超賣）

計算邏輯：把每天的漲跌拆成「漲幅」和「跌幅」兩欄，分別取 14 日平均，再算比值。

In [ ]:
delta = df["Close"].diff()
gain = delta.clip(lower=0)          # 只留漲的部分，跌的變 0
loss = -delta.clip(upper=0)         # 只留跌的部分，取正值

avg_gain = gain.ewm(alpha=1/14, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/14, adjust=False).mean()

rs = avg_gain / avg_loss
df["RSI_14"] = 100 - (100 / (1 + rs))

df[["Date", "Close", "RSI_14"]].tail()

## 預測目標

In [ ]:
# 明天的收盤價
df["Target_Close"] = df["Close"].shift(-1)

# 明天漲(U)還是跌(D)
df["Target_UD"] = (df["Close"].shift(-1) > df["Close"]).mask(df["Close"].shift(-1).isna()).map({True: "U", False: "D"})

df[["Date", "Close", "Target_Close", "Target_UD"]].head()

## 處理缺值：`dropna()`

不少加工在資料開頭或結尾一定會產生 `NaN`，  
例如 `MA_20` 前 19 天算不出來、最後一天沒有「明天」。  
建模前要先把這些列拿掉，不然模型會出錯。

In [ ]:
print("處理前：", df.shape)
df_model = df.dropna().reset_index(drop=True)
print("處理後：", df_model.shape)
df_model.head(3)

# 監督式學習經典模型

## 迴歸（預測明天收盤價）

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn import metrics

feature_cols = ["MA_5", "MA_10", "MA_20", "RSI_14", "DailyReturn", "Volatility_5", "Volume"]

X = df_model[feature_cols]
y = df_model["Target_Close"]

In [ ]:
test_size = 60  # 用最後 60 個交易日當測試集

train_X = X.iloc[:-test_size]
test_X  = X.iloc[-test_size:]

train_y = y.iloc[:-test_size]
test_y  = y.iloc[-test_size:]

print(f"訓練集：{len(train_X)} 筆　測試集：{len(test_X)} 筆")

In [ ]:
reg = LinearRegression()
reg.fit(train_X, train_y)

pred_y = reg.predict(test_X)
print("R²:", reg.score(test_X, test_y))

### 模型：線性回歸

#### [練習] 只用 `MA_5` 和 `Volume` 兩個特徵重新訓練，比較 R Squared 有沒有變化

In [ ]:
feature_cols_v2 = ["MA_5", "Volume"]

X2 = df_model[feature_cols_v2]
train_X2 = X2.iloc[:-test_size]
test_X2  = X2.iloc[-test_size:]

reg2 = LinearRegression()
reg2.fit(train_X2, train_y)
pred_y2 = reg2.predict(test_X2)

print("R²:", reg2.score(test_X2, test_y))

## 分類（預測明天漲跌）

這次 Y 換成 `Target_UD`（明天是漲 U 還是跌 D），X 維持一樣。

In [ ]:
y_cls = df_model["Target_UD"]

train_y_cls = y_cls.iloc[:-test_size]
test_y_cls  = y_cls.iloc[-test_size:]

train_y_cls.value_counts()

### 模型：決策樹

In [ ]:
from sklearn import tree

clf_tree = tree.DecisionTreeClassifier(max_depth=3, random_state=42)
clf_tree.fit(train_X, train_y_cls)

pred_tree = clf_tree.predict(test_X)
acc_tree = metrics.accuracy_score(test_y_cls, pred_tree)
print("決策樹 Accuracy:", acc_tree)

### 模型：kNN

In [ ]:
from sklearn import neighbors

clf_knn = neighbors.KNeighborsClassifier(n_neighbors=5)
clf_knn.fit(train_X, train_y_cls)

pred_knn = clf_knn.predict(test_X)
acc_knn = metrics.accuracy_score(test_y_cls, pred_knn)
print("kNN Accuracy:", acc_knn)

### 準確率判斷：

In [ ]:
baseline_pred = train_y_cls.value_counts().idxmax()  # 訓練集裡出現比較多次的類別
baseline_acc = (test_y_cls == baseline_pred).mean()

print(f"永遠猜 '{baseline_pred}' 的 baseline accuracy：{baseline_acc:.3f}")
print(f"決策樹 accuracy：{acc_tree:.3f}")
print(f"kNN accuracy：{acc_knn:.3f}")

#### [練習] 把 `n_neighbors` 從 5 改成 15，看 kNN accuracy 有沒有變化

In [ ]:
from sklearn import neighbors

clf_knn_v2 = neighbors.KNeighborsClassifier(n_neighbors=15)
clf_knn_v2.fit(train_X, train_y_cls)

pred_knn_v2 = clf_knn_v2.predict(test_X)
print("kNN(k=15) Accuracy:", metrics.accuracy_score(test_y_cls, pred_knn_v2))

#### [練習] 把決策樹的 `max_depth` 從 3 改成 6，比較 accuracy 有沒有變好

In [ ]:
from sklearn import tree

clf_tree_v2 = tree.DecisionTreeClassifier(max_depth=6, random_state=42)
clf_tree_v2.fit(train_X, train_y_cls)

pred_tree_v2 = clf_tree_v2.predict(test_X)
print("決策樹(depth=6) Accuracy:", metrics.accuracy_score(test_y_cls, pred_tree_v2))

### Confusion Matrix（混淆矩陣）

In [ ]:
cm = metrics.confusion_matrix(test_y_cls, pred_tree, labels=["U", "D"])
cm_df = pd.DataFrame(cm, index=["實際 U", "實際 D"], columns=["預測 U", "預測 D"])
cm_df

#### [練習] 印出 kNN（`pred_knn`）的 confusion matrix，跟決策樹比較誰的錯誤集中在哪一格

In [ ]:
cm_knn = metrics.confusion_matrix(test_y_cls, pred_knn, labels=["U", "D"])
cm_knn_df = pd.DataFrame(cm_knn, index=["實際 U", "實際 D"], columns=["預測 U", "預測 D"])
cm_knn_df

# XGBoost

In [ ]:
!pip install xgboost -q

## XGBoost vs 決策樹

In [ ]:
from xgboost import XGBClassifier

train_y_bin = (train_y_cls == "U").astype(int)
test_y_bin  = (test_y_cls == "U").astype(int)

clf_xgb = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
clf_xgb.fit(train_X, train_y_bin)

pred_xgb = clf_xgb.predict(test_X)
acc_xgb = metrics.accuracy_score(test_y_bin, pred_xgb)

print(f"決策樹 accuracy：{acc_tree:.3f}")
print(f"XGBoost accuracy：{acc_xgb:.3f}")
print(f"baseline accuracy：{baseline_acc:.3f}")

## 參數調整

| 參數 | 意思 | 影響 |
|---|---|---|
| `n_estimators` | 要疊幾棵樹 | 越多通常越準，但太多會 overfitting、也變慢 |
| `max_depth` | 每棵樹的深度 | 越深越複雜，容易 overfitting |
| `learning_rate` | 每棵樹修正錯誤的「步伐」大小 | 越小學得越穩，但需要更多棵樹（`n_estimators` 要跟著調高） |


### [練習] 參數調整
把 `n_estimators` 改成 300、`max_depth` 改成 5，看分類 accuracy 變化

In [ ]:
clf_xgb_v2 = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1, random_state=42)
clf_xgb_v2.fit(train_X, train_y_bin)

pred_xgb_v2 = clf_xgb_v2.predict(test_X)
print("XGBoost(n=300, depth=5) Accuracy:", metrics.accuracy_score(test_y_bin, pred_xgb_v2))

# [情境練習]

換一檔股票（例如 AAPL 或 GOOGL），重複一次「特徵工程 → 切分 → 訓練 → 比較」的流程。

## 選定股票

In [ ]:
stock_id = "AAPL"  # 換成 AAPL 或 GOOGL 都可以

df_p = yf.Ticker(stock_id).history(period="5y")
df_p = df_p.reset_index()[["Date", "Open", "High", "Low", "Close", "Volume"]]
df_p["Date"] = df_p["Date"].dt.tz_localize(None)
df_p = df_p.sort_values("Date").reset_index(drop=True)

## 特徵工程

In [ ]:
df_p["DailyReturn"] = df_p["Close"].pct_change()
df_p["Volatility_5"] = df_p["DailyReturn"].rolling(5).std()

df_p["MA_5"] = df_p["Close"].rolling(5).mean()
df_p["MA_20"] = df_p["Close"].rolling(20).mean()

delta = df_p["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
df_p["RSI_14"] = 100 - (100 / (1 + gain.rolling(14).mean() / loss.rolling(14).mean()))

df_p["Target_UD"] = (df_p["Close"].shift(-1) > df_p["Close"]).mask(df["Close"].shift(-1).isna()).map({True: "U", False: "D"})

df_p_model = df_p.dropna()  # 去掉 NaN
df_p_model.info()

## 切分訓練測試資料

In [ ]:
feature_cols_p = ["MA_5", "MA_20", "RSI_14", "DailyReturn", "Volatility_5", "Volume"]

X_p = df_p_model[feature_cols_p]
y_p = df_p_model["Target_UD"]

test_size_p = 60
train_X_p, test_X_p = X_p.iloc[:-test_size_p], X_p.iloc[-test_size_p:]
train_y_p, test_y_p = y_p.iloc[:-test_size_p], y_p.iloc[-test_size_p:]

## 模型比較

In [ ]:
results = {}

clf_tree_p = tree.DecisionTreeClassifier(max_depth=3, random_state=42)
clf_tree_p.fit(train_X_p, train_y_p)
results["Decision Tree"] = metrics.accuracy_score(test_y_p, clf_tree_p.predict(test_X_p))

clf_knn_p = neighbors.KNeighborsClassifier(n_neighbors=5)
clf_knn_p.fit(train_X_p, train_y_p)
results["kNN"] = metrics.accuracy_score(test_y_p, clf_knn_p.predict(test_X_p))

train_y_p_bin = (train_y_p == "U").astype(int)
test_y_p_bin  = (test_y_p == "U").astype(int)
clf_xgb_p = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
clf_xgb_p.fit(train_X_p, train_y_p_bin)
results["XGBoost"] = metrics.accuracy_score(test_y_p_bin, clf_xgb_p.predict(test_X_p))

baseline_p = (test_y_p == train_y_p.value_counts().idxmax()).mean()
results["Baseline"] = baseline_p

pd.Series(results).sort_values(ascending=False)

**討論**：三個模型的準確率跟 baseline 比起來如何？換一檔股票結果會一樣嗎？  
如果時間允許，也可以試著調整 `test_size`、`max_depth`，觀察結果怎麼變化。